In [2]:
from pathlib import Path
from itertools import combinations
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import (train_test_split, KFold, RepeatedKFold, GroupKFold,
StratifiedKFold, cross_validate, cross_val_predict, GridSearchCV)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.mixture import GaussianMixture
from sklearn.inspection import permutation_importance
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
roc_auc_score, brier_score_loss, accuracy_score, confusion_matrix)
import joblib
import json



SEED = 42


rng = np.random.default_rng(SEED)
pd.set_option("display.width", 110); pd.set_option("display.max_columns", 20)
for d in ["figures", "outputs", "model"]:
    Path(d).mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
"axes.spines.top": False, "axes.spines.right": False})
print("Setup is complete")

Setup is complete


Loading the data

In [3]:
import pandas as pd
print(pd.__version__)

3.0.3


In [4]:
from ucimlrepo import fetch_ucirepo

ee = fetch_ucirepo(id = 242)
raw_data = pd.concat([ee.data.features, ee.data.targets], axis = 1)

names = {"X1": "compactness", "X2": "surface_area", "X3": "wall_area", "X4": "roof_area",
         "X5": "height", "X6": "orientation", "X7": "glazing_area", "X8": "glazing_dist",
         "Y1": "heating_load", "Y2": "cooling_load"}
df = raw_data.rename(columns = names)
TARGETS = ["heating_load", "cooling_load"]

print(df.shape)
df.head()

(768, 10)


,compactness,surface_area,wall_area,roof_area,height,orientation,glazing_area,glazing_dist,heating_load,cooling_load
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,15.55,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,15.55,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,15.55,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,15.55,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,20.84,28.28


Checking for duplicate rows and missing values

In [5]:
print("Duplicate Rows:", df.duplicated().sum())
print("Missing Values:", df.isna().sum().sum())

pd.DataFrame({"dtype": df.dtypes,
             "unique": df.nunique(),
             "min": df.min(),
             "max": df.max()
             })

Duplicate Rows: 0
Missing Values: 0


,dtype,unique,min,max
compactness,float64,12,0.62,0.98
surface_area,float64,12,514.50,808.50
wall_area,float64,7,245.00,416.50
roof_area,float64,4,110.25,220.50
height,float64,2,3.50,7.00
orientation,int64,4,2.00,5.00
glazing_area,float64,4,0.00,0.40
glazing_dist,int64,6,0.00,5.00
heating_load,float64,586,6.01,43.10
cooling_load,float64,636,10.90,48.03


Checking relationships between features

In [6]:
print(f"surface = wall + 2*roof: {np.allclose(df.surface_area, df.wall_area + 2 * df.roof_area)}")

print(f"glazing 0 -> dist 0: {(df.loc[df.glazing_area == 0, 'glazing_dist'] == 0).all()}")

print(f"Shapes x orientations x glazing setups: {df.compactness.nunique()} x {df.orientation.nunique()} x {len(df) // 48} = {len(df)}")

df.groupby("compactness")[["surface_area", "wall_area", "roof_area", "height"]].first()


surface = wall + 2*roof: True
glazing 0 -> dist 0: True
Shapes x orientations x glazing setups: 12 x 4 x 16 = 768


,surface_area,wall_area,roof_area,height
compactness,,,,
0.62,808.5,367.5,220.50,3.5
0.64,784.0,343.0,220.50,3.5
0.66,759.5,318.5,220.50,3.5
0.69,735.0,294.0,220.50,3.5
0.71,710.5,269.5,220.50,3.5
0.74,686.0,245.0,220.50,3.5
0.76,661.5,416.5,122.50,7.0
0.79,637.0,343.0,147.00,7.0
0.82,612.5,318.5,147.00,7.0
